<a href="https://colab.research.google.com/github/toche7/MLPython1Day/blob/main/MLResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab ML for Research

แล็บนี้มุ่งเน้นการพัฒนาและประเมินโมเดล Machine Learning สำหรับการทำนายระยะเวลาการเข้าพักในโรงพยาบาล (regression) และการกลับมาเข้ารับการรักษาซ้ำภายใน 30 วัน (classification) โดยใช้ชุดข้อมูลสังเคราะห์และชุดข้อมูลภายนอก

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/toche7/DataSets/refs/heads/main/Readmission.csv'
df = pd.read_csv(url)

print("Data loaded successfully:")
display(df.head())
print(df.info())

---  
เซลล์โค้ดนี้ทำหน้าที่โหลดข้อมูลจากไฟล์ CSV ที่ระบุ URL (`Readmission.csv`) เข้ามาใน Pandas DataFrame ชื่อ `df` และแสดงตัวอย่างข้อมูล 5 แถวแรก พร้อมทั้งแสดงข้อมูลสรุปของ DataFrame (`df.info()`) เพื่อดูประเภทข้อมูลและจำนวนค่าที่ไม่ใช่ค่าว่าง

##  Hospital Readmission Dataset (ข้อมูลจำลอง)

ชุดข้อมูลจำลองนี้มี 500 แถว และ 10 คอลัมน์ โดยมีรายละเอียดของคอลัมน์ดังนี้:

*   `age`: อายุผู้ป่วย (ตัวเลข)
*   `gender`: เพศ (categorical: Male, Female, Other)
*   `num_comorbidities`: จำนวนโรคร่วม (ตัวเลข)
*   `length_of_stay`: วันนอนโรงพยาบาล (ตัวเลข)
*   `readmission_30d`: กลับมารักษาซ้ำใน 30 วัน (binary: 0/1)
*   `admission_type`: ประเภทการเข้ารับการรักษา (categorical: Emergency, Elective, Urgent)
*   `severity_score`: คะแนนความรุนแรงของโรค (ตัวเลข)
*   `medication_count`: จำนวนยาที่ได้รับ (ตัวเลข)
*   `diagnosis_code`: รหัสการวินิจฉัย (categorical)
*   `hospital_id`: รหัสโรงพยาบาล (ตัวเลข)


## Lab Part 1: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("EDA Report for DataFrame 'df':\n")

# 1. Display basic information and shape
print("DataFrame Info:")
df.info()
print(f"\nDataFrame Shape: {df.shape}\n")

# 2. Display descriptive statistics for numerical columns
print("Descriptive Statistics for Numerical Columns:")
display(df.describe())

# 3. Check for missing values
print("\nMissing Values:")
display(df.isnull().sum())

# 4. Visualize distributions of numerical features
print("\nVisualizing Distributions of Numerical Features:")
# Select numerical columns for visualization
# Exclude 'hospital_id' if it's more like a categorical identifier, though it's int64
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'hospital_id' in numerical_cols:
    numerical_cols.remove('hospital_id') # Often hospital_id is treated as categorical

plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # Adjust subplot grid as needed
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

# 5. Visualize distributions of categorical features
print("\nVisualizing Distributions of Categorical Features:")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Add hospital_id back if it was removed from numerical and is truly categorical
if 'hospital_id' not in categorical_cols and 'hospital_id' in df.columns:
    categorical_cols.append('hospital_id')


plt.figure(figsize=(15, 8))
for i, col in enumerate(categorical_cols):
    plt.subplot(2, 2, i + 1) # Adjust subplot grid as needed
    sns.countplot(data=df, x=col, palette='viridis')
    plt.title(f'Count of {col}')
    plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# 6. Correlation matrix for numerical features
print("\nCorrelation Matrix of Numerical Features:")
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

## Lab Part 2: Regression

### Data Preprocessing และ Model Training สำหรับ `length_of_stay` (Linear Regression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Define features (X) and target (y) for length_of_stay regression
X_los = df.drop(['length_of_stay', 'readmission_30d'], axis=1) # Drop both targets
y_los = df['length_of_stay'] # Set 'length_of_stay' as the new target

# Convert categorical features to numerical using one-hot encoding
X_los = pd.get_dummies(X_los, columns=['gender', 'admission_type', 'diagnosis_code'], drop_first=True)

# Split data into training and testing sets
X_train_los, X_test_los, y_train_los, y_test_los = train_test_split(X_los, y_los, test_size=0.3, random_state=42)

print("Data Preprocessing Complete for length_of_stay.")
print(f"X_train_los shape: {X_train_los.shape}")
print(f"y_train_los shape: {y_train_los.shape}")

In [ ]:
X_los

### Training Linear Regression Model for `length_of_stay`

In [ ]:
# Initialize and train a Linear Regression model
model_lr_los = LinearRegression()
model_lr_los.fit(X_train_los, y_train_los)

print("Linear Regression Model Trained.")

### Evaluating Linear Regression Model Performance

In [ ]:
# Make predictions on the test set
y_pred_lr_los = model_lr_los.predict(X_test_los)

# Evaluate the model
mae_lr_los = mean_absolute_error(y_test_los, y_pred_lr_los)
r2_lr_los = r2_score(y_test_los, y_pred_lr_los)

print(f"Linear Regression (Length of Stay) - Mean Absolute Error (MAE): {mae_lr_los:.2f}")
print(f"Linear Regression (Length of Stay) - R-squared (R²): {r2_lr_los:.2f}")

## Lab Part 3 - Classification

### Data Preprocessing และ Model Training สำหรับ `readmission_30d` (Logistic Regression)

---  
*   **Cell `154e3c9a`**: เซลล์ข้อความนี้ระบุหัวข้อว่าเป็นการเตรียมข้อมูลและการฝึกโมเดลสำหรับ `readmission_30d` โดยใช้ Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Define features (X) and target (y) for readmission_30d classification
X_read = df.drop(['length_of_stay', 'readmission_30d'], axis=1) # Drop both targets
y_read = df['readmission_30d'] # Set 'readmission_30d' as the new target

# Convert categorical features to numerical using one-hot encoding
X_read = pd.get_dummies(X_read, columns=['gender', 'admission_type', 'diagnosis_code'], drop_first=True)

# Split data into training and testing sets
X_train_read, X_test_read, y_train_read, y_test_read = train_test_split(X_read, y_read, test_size=0.3, random_state=42, stratify=y_read)

print("Data Preprocessing Complete for readmission_30d.")
print(f"X_train_read shape: {X_train_read.shape}")
print(f"y_train_read shape: {y_train_read.shape}")

### Training Logistic Regression Model for `readmission_30d`

In [ ]:
# Initialize and train a Logistic Regression model
# Increased max_iter to ensure convergence, as the previous model had a ConvergenceWarning
model_lr_read = LogisticRegression(random_state=42, max_iter=1000)
model_lr_read.fit(X_train_read, y_train_read)

print("Logistic Regression Model Trained.")

### Evaluating Logistic Regression Model Performance

In [ ]:
# Make predictions on the test set
y_pred_lr_read = model_lr_read.predict(X_test_read)
y_prob_lr_read = model_lr_read.predict_proba(X_test_read)[:, 1] # Probabilities for ROC AUC

# Evaluate the model
accuracy_lr_read = accuracy_score(y_test_read, y_pred_lr_read)
precision_lr_read = precision_score(y_test_read, y_pred_lr_read)
recall_lr_read = recall_score(y_test_read, y_pred_lr_read)
f1_lr_read = f1_score(y_test_read, y_pred_lr_read)
roc_auc_lr_read = roc_auc_score(y_test_read, y_prob_lr_read)

print(f"Logistic Regression (Readmission) - Accuracy: {accuracy_lr_read:.2f}")
print(f"Logistic Regression (Readmission) - Precision: {precision_lr_read:.2f}")
print(f"Logistic Regression (Readmission) - Recall: {recall_lr_read:.2f}")
print(f"Logistic Regression (Readmission) - F1-Score: {f1_lr_read:.2f}")
print(f"Logistic Regression (Readmission) - ROC AUC Score: {roc_auc_lr_read:.2f}")